In [1]:
!pip install kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 3.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Tải dataset

In [9]:
import os
import shutil
import kagglehub

# 1. Chỉ định thư mục lưu trữ
target_folder = "/app/data"

# 2. Định nghĩa datasets cần tải
datasets = [
    {
        "name": "rtatman/handwritten-mathematical-expressions",
        "target": os.path.join(target_folder, "CROHME")
    },
    {
        "name": "shahrukhkhan/im2latex100k",
        "target": os.path.join(target_folder, "IM2LATEX")
    }
]

# 3. Tải từng dataset nếu cần
for dataset in datasets:
    dataset_name = dataset["name"]
    target_path = dataset["target"]
    
    # Kiểm tra xem thư mục có tồn tại và không trống
    should_download = False
    
    if not os.path.exists(target_path):
        should_download = True
        print(f"📁 Thư mục {os.path.basename(target_path)} chưa tồn tại")
    elif not os.listdir(target_path):
        should_download = True
        print(f"📁 Thư mục {os.path.basename(target_path)} đang trống")
    else:
        print(f"✓ Dataset {os.path.basename(target_path)} đã tồn tại, bỏ qua tải")
    
    if should_download:
        print(f"⬇️  Đang tải {dataset_name}...")
        
        # Tải dataset
        cache_path = kagglehub.dataset_download(dataset_name)
        
        # Tạo thư mục đích nếu chưa có
        os.makedirs(target_path, exist_ok=True)
        
        # Move tất cả nội dung từ cache vào thư mục đích
        for item in os.listdir(cache_path):
            src = os.path.join(cache_path, item)
            dst = os.path.join(target_path, item)
            
            if os.path.exists(dst):
                if os.path.isdir(dst):
                    shutil.rmtree(dst)
                else:
                    os.remove(dst)
            
            shutil.move(src, dst)
        
        print(f"✓ Hoàn tất tải {os.path.basename(target_path)}")

print("\n🎉 Tất cả datasets đã sẵn sàng!")

✓ Dataset CROHME đã tồn tại, bỏ qua tải
✓ Dataset IM2LATEX đã tồn tại, bỏ qua tải

🎉 Tất cả datasets đã sẵn sàng!


## Xử lý CROHME

In [10]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import defaultdict

def analyze_inkml_file(file_path):
    """Phân tích file InkML để kiểm tra có ground truth không"""
    try:
        tree = ET.parse(file_path)
        root = tree.getroot()
        
        # Tìm annotation truth
        has_truth = False
        has_mathml = False
        has_segmentation = False
        
        for annotation in root.findall('.//{http://www.w3.org/2003/InkML}annotation'):
            if annotation.get('type') == 'truth':
                has_truth = True
                break
        
        # Tìm annotationXML (MathML)
        for ann_xml in root.findall('.//{http://www.w3.org/2003/InkML}annotationXML'):
            if ann_xml.get('type') == 'truth':
                has_mathml = True
                break
        
        # Tìm traceGroup (segmentation)
        for tg in root.findall('.//{http://www.w3.org/2003/InkML}traceGroup'):
            for ann in tg.findall('.//{http://www.w3.org/2003/InkML}annotation'):
                if ann.get('type') == 'truth':
                    has_segmentation = True
                    break
            if has_segmentation:
                break
        
        # Đếm số trace
        num_traces = len(root.findall('.//{http://www.w3.org/2003/InkML}trace'))
        
        return {
            'has_truth': has_truth,
            'has_mathml': has_mathml,
            'has_segmentation': has_segmentation,
            'num_traces': num_traces,
            'status': 'ok'
        }
    except Exception as e:
        return {
            'has_truth': False,
            'has_mathml': False,
            'has_segmentation': False,
            'num_traces': 0,
            'status': f'error: {str(e)}'
        }

def scan_directory(dir_path, max_depth=3, current_depth=0):
    """Quét thư mục đệ quy và phân tích cấu trúc"""
    result = {
        'path': str(dir_path),
        'name': dir_path.name,
        'type': 'directory',
        'subdirs': [],
        'files': {
            'total': 0,
            'inkml': 0,
            'with_truth': 0,
            'without_truth': 0,
            'with_mathml': 0,
            'with_segmentation': 0,
            'other': 0
        },
        'file_list': []
    }
    
    if not dir_path.exists() or not dir_path.is_dir():
        return result
    
    # Quét tất cả items trong thư mục
    items = sorted(list(dir_path.iterdir()))
    
    for item in items:
        if item.is_file():
            result['files']['total'] += 1
            
            if item.suffix.lower() == '.inkml':
                result['files']['inkml'] += 1
                
                # Phân tích file InkML
                analysis = analyze_inkml_file(item)
                
                file_info = {
                    'name': item.name,
                    'size': item.stat().st_size,
                    **analysis
                }
                result['file_list'].append(file_info)
                
                if analysis['has_truth']:
                    result['files']['with_truth'] += 1
                else:
                    result['files']['without_truth'] += 1
                
                if analysis['has_mathml']:
                    result['files']['with_mathml'] += 1
                
                if analysis['has_segmentation']:
                    result['files']['with_segmentation'] += 1
            else:
                result['files']['other'] += 1
        
        elif item.is_dir() and current_depth < max_depth:
            # Đệ quy quét thư mục con
            subdir_result = scan_directory(item, max_depth, current_depth + 1)
            result['subdirs'].append(subdir_result)
    
    return result

def print_tree(data, indent=0, show_files=False):
    """In cấu trúc thư mục dạng cây"""
    prefix = "  " * indent
    icon = "📁" if data['type'] == 'directory' else "📄"
    
    # In tên thư mục/file
    print(f"{prefix}{icon} {data['name']}/")
    
    # In thống kê files
    files = data['files']
    if files['total'] > 0:
        print(f"{prefix}  ├─ 📊 Tổng số files: {files['total']}")
        if files['inkml'] > 0:
            print(f"{prefix}  ├─ ✍️  InkML files: {files['inkml']}")
            print(f"{prefix}  │   ├─ ✓ Có ground truth: {files['with_truth']} ({files['with_truth']/files['inkml']*100:.1f}%)")
            print(f"{prefix}  │   ├─ ✗ Không có truth: {files['without_truth']} ({files['without_truth']/files['inkml']*100:.1f}%)")
            print(f"{prefix}  │   ├─ 📐 Có MathML: {files['with_mathml']}")
            print(f"{prefix}  │   └─ 🔤 Có segmentation: {files['with_segmentation']}")
        if files['other'] > 0:
            print(f"{prefix}  └─ 📋 Files khác: {files['other']}")
    
    # In danh sách files nếu yêu cầu
    if show_files and data['file_list']:
        print(f"{prefix}  📝 Files:")
        for i, f in enumerate(data['file_list'][:5]):  # Chỉ hiện 5 files đầu
            status = "✓" if f['has_truth'] else "✗"
            print(f"{prefix}     {status} {f['name']} ({f['num_traces']} traces)")
        if len(data['file_list']) > 5:
            print(f"{prefix}     ... và {len(data['file_list']) - 5} files khác")
    
    # In thư mục con
    for subdir in data['subdirs']:
        print()
        print_tree(subdir, indent + 1, show_files)

def generate_summary(data):
    """Tạo báo cáo tổng hợp"""
    total_files = data['files']['total']
    total_inkml = data['files']['inkml']
    total_with_truth = data['files']['with_truth']
    total_without_truth = data['files']['without_truth']
    
    for subdir in data['subdirs']:
        sub_summary = generate_summary(subdir)
        total_files += sub_summary['total_files']
        total_inkml += sub_summary['total_inkml']
        total_with_truth += sub_summary['total_with_truth']
        total_without_truth += sub_summary['total_without_truth']
    
    return {
        'total_files': total_files,
        'total_inkml': total_inkml,
        'total_with_truth': total_with_truth,
        'total_without_truth': total_without_truth
    }

# ====================
# CHẠY PHÂN TÍCH
# ====================

crohme_path = Path("/app/data/CROHME")

print("🔍 BẮT ĐẦU QUÉT DATASET CROHME...")
print("=" * 80)
print()

# Quét toàn bộ cấu trúc
result = scan_directory(crohme_path, max_depth=3)

# In cấu trúc cây
print("📂 CẤU TRÚC THƯ MỤC:")
print("=" * 80)
print_tree(result, show_files=False)

print()
print("=" * 80)
print("📊 TỔNG HỢP:")
print("=" * 80)

summary = generate_summary(result)
print(f"📁 Tổng số files: {summary['total_files']}")
print(f"✍️  InkML files: {summary['total_inkml']}")
print(f"  ├─ ✓ Có ground truth: {summary['total_with_truth']} ({summary['total_with_truth']/summary['total_inkml']*100:.1f}%)")
print(f"  └─ ✗ Không có truth: {summary['total_without_truth']} ({summary['total_without_truth']/summary['total_inkml']*100:.1f}%)")

print()
print("✅ HOÀN TẤT PHÂN TÍCH!")

🔍 BẮT ĐẦU QUÉT DATASET CROHME...

📂 CẤU TRÚC THƯ MỤC:
📁 CROHME/

  📁 CROHME_test_2011/
    ├─ 📊 Tổng số files: 348
    ├─ ✍️  InkML files: 348
    │   ├─ ✓ Có ground truth: 0 (0.0%)
    │   ├─ ✗ Không có truth: 348 (100.0%)
    │   ├─ 📐 Có MathML: 0
    │   └─ 🔤 Có segmentation: 0

  📁 CROHME_training_2011/
    ├─ 📊 Tổng số files: 921
    ├─ ✍️  InkML files: 921
    │   ├─ ✓ Có ground truth: 921 (100.0%)
    │   ├─ ✗ Không có truth: 0 (0.0%)
    │   ├─ 📐 Có MathML: 921
    │   └─ 🔤 Có segmentation: 921

  📁 MatricesTest2014/
    ├─ 📊 Tổng số files: 123
    ├─ ✍️  InkML files: 122
    │   ├─ ✓ Có ground truth: 122 (100.0%)
    │   ├─ ✗ Không có truth: 0 (0.0%)
    │   ├─ 📐 Có MathML: 122
    │   └─ 🔤 Có segmentation: 122
    └─ 📋 Files khác: 1

    📁 MatricesTest/
      ├─ 📊 Tổng số files: 123
      ├─ ✍️  InkML files: 122
      │   ├─ ✓ Có ground truth: 122 (100.0%)
      │   ├─ ✗ Không có truth: 0 (0.0%)
      │   ├─ 📐 Có MathML: 122
      │   └─ 🔤 Có segmentation: 122
      └─ 📋 File

In [ ]:
import os
import xml.etree.ElementTree as ET
from pathlib import Path
from PIL import Image, ImageDraw
import pandas as pd
from tqdm import tqdm

def parse_inkml_traces(inkml_file):
    """Parse InkML file và trích xuất traces"""
    try:
        tree = ET.parse(inkml_file)
        root = tree.getroot()
        ns = {'ink': 'http://www.w3.org/2003/InkML'}
        
        traces = []
        for trace in root.findall('.//ink:trace', ns):
            points = []
            trace_data = trace.text.strip() if trace.text else ""
            for point in trace_data.split(','):
                coords = point.strip().split()
                if len(coords) == 2:
                    try:
                        x, y = float(coords[0]), float(coords[1])
                        points.append((x, y))
                    except ValueError:
                        continue
            if points:
                traces.append(points)
        
        return traces
    except Exception as e:
        print(f"Lỗi parse {inkml_file}: {e}")
        return []

def get_ground_truth(inkml_file):
    """Lấy ground truth từ file InkML"""
    try:
        tree = ET.parse(inkml_file)
        root = tree.getroot()
        ns = {'ink': 'http://www.w3.org/2003/InkML'}
        
        for annotation in root.findall('.//ink:annotation', ns):
            if annotation.get('type') == 'truth':
                truth = annotation.text.strip() if annotation.text else ""
                # Loại bỏ $ ở đầu và cuối nếu có
                truth = truth.strip('$').strip()
                return truth
        
        return None
    except Exception as e:
        print(f"Lỗi lấy truth {inkml_file}: {e}")
        return None

def traces_to_image(traces, img_size=(400, 400), padding=20, line_width=2):
    """Chuyển đổi traces thành ảnh"""
    if not traces:
        return None
    
    try:
        # Tìm bounding box
        all_points = [p for trace in traces for p in trace]
        if not all_points:
            return None
        
        xs = [p[0] for p in all_points]
        ys = [p[1] for p in all_points]
        
        min_x, max_x = min(xs), max(xs)
        min_y, max_y = min(ys), max(ys)
        
        # Tránh chia cho 0
        width = max_x - min_x
        height = max_y - min_y
        
        if width == 0 or height == 0:
            return None
        
        # Tính scale để fit vào ảnh
        scale = min((img_size[0] - 2*padding) / width, 
                    (img_size[1] - 2*padding) / height)
        
        # Tạo ảnh trắng
        img = Image.new('RGB', img_size, 'white')
        draw = ImageDraw.Draw(img)
        
        # Vẽ từng trace
        for trace in traces:
            if len(trace) < 2:
                continue
            
            # Scale và translate points
            scaled_trace = [
                (
                    (x - min_x) * scale + padding,
                    (y - min_y) * scale + padding
                )
                for x, y in trace
            ]
            
            # Vẽ đường nối
            for i in range(len(scaled_trace) - 1):
                draw.line([scaled_trace[i], scaled_trace[i+1]], 
                         fill='black', width=line_width)
        
        return img
    except Exception as e:
        print(f"Lỗi tạo ảnh: {e}")
        return None

def process_crohme_dataset(crohme_path):
    """Xử lý dataset CROHME và tạo cấu trúc mới"""
    
    crohme_path = Path(crohme_path)
    
    # Tạo thư mục đích
    ground_truth_dir = crohme_path / "ground_truth"
    non_ground_truth_dir = crohme_path / "non_ground_truth"
    
    gt_images_dir = ground_truth_dir / "images"
    ngt_images_dir = non_ground_truth_dir / "images"
    
    # Tạo thư mục nếu chưa có
    gt_images_dir.mkdir(parents=True, exist_ok=True)
    ngt_images_dir.mkdir(parents=True, exist_ok=True)
    
    # Danh sách để lưu vào CSV
    gt_data = []  # [(formula, image_name), ...]
    ngt_data = []  # [image_name, ...]
    
    # Tìm tất cả file InkML
    inkml_files = list(crohme_path.rglob("*.inkml"))
    
    # Lọc bỏ các file trong ground_truth và non_ground_truth để tránh đệ quy
    inkml_files = [f for f in inkml_files 
                   if "ground_truth" not in str(f) and "non_ground_truth" not in str(f)]
    
    print(f"🔍 Tìm thấy {len(inkml_files)} file InkML")
    print("🚀 Bắt đầu xử lý...\n")
    
    # Đếm số lượng
    gt_count = 0
    ngt_count = 0
    error_count = 0
    
    # Xử lý từng file
    for inkml_file in tqdm(inkml_files, desc="Xử lý files"):
        # Parse traces
        traces = parse_inkml_traces(inkml_file)
        if not traces:
            error_count += 1
            continue
        
        # Tạo tên file ảnh (dùng tên file gốc)
        image_name = inkml_file.stem + ".png"
        
        # Tạo ảnh
        img = traces_to_image(traces)
        if img is None:
            error_count += 1
            continue
        
        # Kiểm tra có ground truth không
        truth = get_ground_truth(inkml_file)
        
        if truth:
            # Có ground truth
            img_path = gt_images_dir / image_name
            img.save(img_path)
            gt_data.append({
                'formula': truth,
                'image': image_name
            })
            gt_count += 1
        else:
            # Không có ground truth
            img_path = ngt_images_dir / image_name
            img.save(img_path)
            ngt_data.append({
                'image': image_name
            })
            ngt_count += 1
    
    # Lưu CSV cho ground truth
    if gt_data:
        gt_df = pd.DataFrame(gt_data)
        gt_csv_path = ground_truth_dir / "dataset.csv"
        gt_df.to_csv(gt_csv_path, index=False, encoding='utf-8')
        print(f"\n✓ Đã lưu {len(gt_data)} mẫu có ground truth vào {gt_csv_path}")
    
    # Lưu CSV cho non ground truth
    if ngt_data:
        ngt_df = pd.DataFrame(ngt_data)
        ngt_csv_path = non_ground_truth_dir / "dataset.csv"
        ngt_df.to_csv(ngt_csv_path, index=False, encoding='utf-8')
        print(f"✓ Đã lưu {len(ngt_data)} mẫu không có ground truth vào {ngt_csv_path}")
    
    # Báo cáo tổng hợp
    print("\n" + "="*80)
    print("📊 TỔNG HỢP KẾT QUẢ:")
    print("="*80)
    print(f"✓ Có ground truth: {gt_count} files")
    print(f"✗ Không có ground truth: {ngt_count} files")
    print(f"⚠️  Lỗi/Bỏ qua: {error_count} files")
    print(f"📁 Tổng cộng: {len(inkml_files)} files")
    print("="*80)
    
    return {
        'ground_truth': gt_count,
        'non_ground_truth': ngt_count,
        'errors': error_count,
        'total': len(inkml_files)
    }

# ====================
# CHẠY XỬ LÝ
# ====================

# Cài đặt thư viện cần thiết
try:
    from PIL import Image, ImageDraw
    import pandas as pd
    from tqdm import tqdm
except ImportError:
    print("Chưa cài đặt thư viện")
# Chạy xử lý
crohme_path = "/app/data/CROHME"
results = process_crohme_dataset(crohme_path)

🔍 Tìm thấy 24203 file InkML
🚀 Bắt đầu xử lý...



Xử lý files:  54%|█████▍    | 13132/24203 [05:25<01:26, 127.97it/s]

Lỗi parse /app/data/CROHME/TrainINKML_2013/MfrDB0104.inkml: not well-formed (invalid token): line 15, column 23


Xử lý files:  59%|█████▉    | 14347/24203 [05:34<01:31, 107.86it/s]

Lỗi parse /app/data/CROHME/TrainINKML_2013/MfrDB3088.inkml: no element found: line 1, column 0


...

Lỗi parse /app/data/CROHME/TrainINKML_2013/TrainINKML/MfrDB/MfrDB0104.inkml: not well-formed (invalid token): line 15, column 23


Xử lý files:  99%|█████████▊| 23859/24203 [09:53<00:04, 85.09it/s] 

Lỗi parse /app/data/CROHME/TrainINKML_2013/TrainINKML/MfrDB/MfrDB3088.inkml: no element found: line 1, column 0


Xử lý files: 100%|██████████| 24203/24203 [09:56<00:00, 40.56it/s] 



✓ Đã lưu 18855 mẫu có ground truth vào /app/data/CROHME/ground_truth/dataset.csv
✓ Đã lưu 2178 mẫu không có ground truth vào /app/data/CROHME/non_ground_truth/dataset.csv

📊 TỔNG HỢP KẾT QUẢ:
✓ Có ground truth: 18855 files
✗ Không có ground truth: 2178 files
⚠️  Lỗi/Bỏ qua: 3170 files
📁 Tổng cộng: 24203 files

📝 Mẫu ground truth (5 dòng đầu):
                                             formula  \
0                                            \phi(x)   
1  \log_c(a - b) = \log_c(c^{(\log_ca - \log_cb)}...   
2                                151 \pm 143 \div 97   
3           21-5\sqrt{21} + (15\sqrt{7}-21\sqrt{3})i   
4                                 78 \pm 5 \times 47   

                           image  
0  formulaire001-equation001.png  
1  formulaire001-equation003.png  
2  formulaire001-equation010.png  
3  formulaire001-equation011.png  
4  formulaire001-equation015.png  

📝 Mẫu non ground truth (5 dòng đầu):
                           image
0                     algb02.png
1  